# Unsupervised Topic Modeling Pipeline

A single-file notebook for the six project stages: acquire, preprocess, represent, train, evaluate, and infer. The implementation for every stage is included below, so you can run or share this notebook without importing the project's `src/` modules.

> **Dataset:** this project uses the professor-recommended 20 Newsgroups corpus, preserving the official train/test split and the 20 original categories.

Run the notebook in the project environment after installing `requirements.txt` and the spaCy model `en_core_web_sm`.

## 0. Setup and run settings

This notebook follows the professor-recommended 20 Newsgroups corpus. Run all six stages in order; outputs are written into `data/` and `artifacts/`.

In [1]:
from pathlib import Path
import sys
import json

# Locate the repository when the notebook is opened from notebooks/ or the root.
PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project folder (it must contain both data/ and artifacts/). "
        "Open the notebook from the project folder or one of its subfolders."
    )

# The project follows the professor's recommendation: the 20 Newsgroups corpus.
DATA_SOURCE = "20newsgroups"

# Human-readable interpretations for all 60 learned topics.
INTERPRETATIONS = {'LDA_BoW (Controlled & Practical)': {'god, jesus, church, bible, christ, christian, life, believe, word, law': {'label': 'Christianity '
                                                                                                                          'and '
                                                                                                                          'religious '
                                                                                                                          'life',
                                                                                                                 'rationale': 'God, '
                                                                                                                              'Jesus, '
                                                                                                                              'church, '
                                                                                                                              'Bible, '
                                                                                                                              'Christ, '
                                                                                                                              'and '
                                                                                                                              'Christian '
                                                                                                                              'beliefs '
                                                                                                                              'are '
                                                                                                                              'the '
                                                                                                                              'dominant '
                                                                                                                              'terms; '
                                                                                                                              'law '
                                                                                                                              'and '
                                                                                                                              'life '
                                                                                                                              'add '
                                                                                                                              'broad '
                                                                                                                              'discussion '
                                                                                                                              'context.'},
                                      'power, ground, wire, light, water, circuit, cable, hole, supply, tape': {'label': 'Electrical '
                                                                                                                         'circuits '
                                                                                                                         'and '
                                                                                                                         'power '
                                                                                                                         'systems',
                                                                                                                'rationale': 'Wire, '
                                                                                                                             'circuit, '
                                                                                                                             'current, '
                                                                                                                             'light, '
                                                                                                                             'and '
                                                                                                                             'supply '
                                                                                                                             'vocabulary '
                                                                                                                             'points '
                                                                                                                             'to '
                                                                                                                             'electrical '
                                                                                                                             'equipment '
                                                                                                                             'and '
                                                                                                                             'troubleshooting.'},
                                      'god, believe, argument, evidence, atheist, exist, belief, claim, religion, true': {'label': 'Atheism '
                                                                                                                                   'and '
                                                                                                                                   'arguments '
                                                                                                                                   'about '
                                                                                                                                   'belief',
                                                                                                                          'rationale': 'Atheist, '
                                                                                                                                       'evidence, '
                                                                                                                                       'existence, '
                                                                                                                                       'belief, '
                                                                                                                                       'and '
                                                                                                                                       'argument '
                                                                                                                                       'signal '
                                                                                                                                       'debate '
                                                                                                                                       'about '
                                                                                                                                       'religion '
                                                                                                                                       'and '
                                                                                                                                       'the '
                                                                                                                                       'existence '
                                                                                                                                       'of '
                                                                                                                                       'God.'},
                                      'drive, card, disk, system, driver, need, monitor, video, hard, mac': {'label': 'Computer '
                                                                                                                      'hardware '
                                                                                                                      'and '
                                                                                                                      'storage',
                                                                                                             'rationale': 'Drives, '
                                                                                                                          'disks, '
                                                                                                                          'cards, '
                                                                                                                          'drivers, '
                                                                                                                          'monitors, '
                                                                                                                          'and '
                                                                                                                          'video '
                                                                                                                          'identify '
                                                                                                                          'personal-computer '
                                                                                                                          'hardware '
                                                                                                                          'discussions.'},
                                      'president, government, program, report, state, technology, national, child, administration, health': {'label': 'Government, '
                                                                                                                                                      'public '
                                                                                                                                                      'programs, '
                                                                                                                                                      'and '
                                                                                                                                                      'policy '
                                                                                                                                                      '(mixed)',
                                                                                                                                             'rationale': 'President, '
                                                                                                                                                          'government, '
                                                                                                                                                          'state, '
                                                                                                                                                          'and '
                                                                                                                                                          'administration '
                                                                                                                                                          'anchor '
                                                                                                                                                          'a '
                                                                                                                                                          'policy '
                                                                                                                                                          'topic, '
                                                                                                                                                          'while '
                                                                                                                                                          'health '
                                                                                                                                                          'and '
                                                                                                                                                          'child '
                                                                                                                                                          'indicate '
                                                                                                                                                          'related '
                                                                                                                                                          'public-affairs '
                                                                                                                                                          'coverage.'},
                                      'car, bike, engine, buy, ride, drive, road, speed, mile, tire': {'label': 'Cars, '
                                                                                                                'motorcycles, '
                                                                                                                'and '
                                                                                                                'vehicle '
                                                                                                                'maintenance',
                                                                                                       'rationale': 'Cars, '
                                                                                                                    'bikes, '
                                                                                                                    'engines, '
                                                                                                                    'roads, '
                                                                                                                    'tires, '
                                                                                                                    'and '
                                                                                                                    'mileage '
                                                                                                                    'describe '
                                                                                                                    'vehicle '
                                                                                                                    'ownership '
                                                                                                                    'and '
                                                                                                                    'use.'},
                                      'company, apr, pay, month, trial, logo, organization, test, advertising, million': {'label': 'Company '
                                                                                                                                   'and '
                                                                                                                                   'commercial '
                                                                                                                                   'information '
                                                                                                                                   '(mixed)',
                                                                                                                          'rationale': 'Company, '
                                                                                                                                       'payment, '
                                                                                                                                       'trial, '
                                                                                                                                       'logo, '
                                                                                                                                       'and '
                                                                                                                                       'advertising '
                                                                                                                                       'terms '
                                                                                                                                       'suggest '
                                                                                                                                       'business '
                                                                                                                                       'notices, '
                                                                                                                                       'though '
                                                                                                                                       'the '
                                                                                                                                       'topic '
                                                                                                                                       'is '
                                                                                                                                       'not '
                                                                                                                                       'tightly '
                                                                                                                                       'focused.'},
                                      'space, launch, system, datum, nasa, information, satellite, include, university, research': {'label': 'Space '
                                                                                                                                             'exploration '
                                                                                                                                             'and '
                                                                                                                                             'NASA '
                                                                                                                                             'research',
                                                                                                                                    'rationale': 'Space, '
                                                                                                                                                 'launch, '
                                                                                                                                                 'NASA, '
                                                                                                                                                 'satellites, '
                                                                                                                                                 'and '
                                                                                                                                                 'research '
                                                                                                                                                 'identify '
                                                                                                                                                 'aerospace '
                                                                                                                                                 'and '
                                                                                                                                                 'space-science '
                                                                                                                                                 'discussion; '
                                                                                                                                                 'information-system '
                                                                                                                                                 'terms '
                                                                                                                                                 'are '
                                                                                                                                                 'secondary '
                                                                                                                                                 'noise.'},
                                      'scsi, max, bit, fast, university, mode, ide, mac, cost, book': {'label': 'Computer '
                                                                                                                'interfaces '
                                                                                                                'and '
                                                                                                                'hardware '
                                                                                                                '(mixed)',
                                                                                                       'rationale': 'SCSI, '
                                                                                                                    'IDE, '
                                                                                                                    'bit, '
                                                                                                                    'mode, '
                                                                                                                    'and '
                                                                                                                    'Mac '
                                                                                                                    'suggest '
                                                                                                                    'hardware; '
                                                                                                                    'university, '
                                                                                                                    'cost, '
                                                                                                                    'and '
                                                                                                                    'book '
                                                                                                                    'make '
                                                                                                                    'this '
                                                                                                                    'topic '
                                                                                                                    'less '
                                                                                                                    'specific.'},
                                      'armenian, turkish, war, government, kill, turkey, nazi, muslim, greek, turk': {'label': 'Armenian '
                                                                                                                               'and '
                                                                                                                               'Turkish '
                                                                                                                               'history '
                                                                                                                               'and '
                                                                                                                               'conflict',
                                                                                                                      'rationale': 'Armenian, '
                                                                                                                                   'Turkish, '
                                                                                                                                   'Turkey, '
                                                                                                                                   'Greek, '
                                                                                                                                   'war, '
                                                                                                                                   'and '
                                                                                                                                   'genocide-related '
                                                                                                                                   'vocabulary '
                                                                                                                                   'identify '
                                                                                                                                   'historical '
                                                                                                                                   'and '
                                                                                                                                   'political '
                                                                                                                                   'conflict.'},
                                      'san, period, chicago, york, los, city, angeles, los angeles, drug, pittsburgh': {'label': 'US '
                                                                                                                                 'cities '
                                                                                                                                 'and '
                                                                                                                                 'local '
                                                                                                                                 'references '
                                                                                                                                 '(mixed)',
                                                                                                                        'rationale': 'San '
                                                                                                                                     'Francisco, '
                                                                                                                                     'Chicago, '
                                                                                                                                     'New '
                                                                                                                                     'York, '
                                                                                                                                     'Los '
                                                                                                                                     'Angeles, '
                                                                                                                                     'and '
                                                                                                                                     'Pittsburgh '
                                                                                                                                     'are '
                                                                                                                                     'place '
                                                                                                                                     'names. '
                                                                                                                                     'The '
                                                                                                                                     'remaining '
                                                                                                                                     'terms '
                                                                                                                                     'are '
                                                                                                                                     'broad '
                                                                                                                                     'and '
                                                                                                                                     'do '
                                                                                                                                     'not '
                                                                                                                                     'establish '
                                                                                                                                     'one '
                                                                                                                                     'narrow '
                                                                                                                                     'subject, '
                                                                                                                                     'so '
                                                                                                                                     'this '
                                                                                                                                     'topic '
                                                                                                                                     'is '
                                                                                                                                     'best '
                                                                                                                                     'treated '
                                                                                                                                     'as '
                                                                                                                                     'a '
                                                                                                                                     'mixed '
                                                                                                                                     'collection '
                                                                                                                                     'of '
                                                                                                                                     'location-related '
                                                                                                                                     'posts.'},
                                      'patient, disease, food, cause, medical, doctor, pain, eat, treatment, msg': {'label': 'Medicine, '
                                                                                                                             'health, '
                                                                                                                             'and '
                                                                                                                             'nutrition',
                                                                                                                    'rationale': 'Patient, '
                                                                                                                                 'disease, '
                                                                                                                                 'medical '
                                                                                                                                 'care, '
                                                                                                                                 'doctors, '
                                                                                                                                 'pain, '
                                                                                                                                 'and '
                                                                                                                                 'treatment '
                                                                                                                                 'form '
                                                                                                                                 'the '
                                                                                                                                 'core '
                                                                                                                                 'health '
                                                                                                                                 'theme; '
                                                                                                                                 'food, '
                                                                                                                                 'eating, '
                                                                                                                                 'and '
                                                                                                                                 'MSG '
                                                                                                                                 'add '
                                                                                                                                 'a '
                                                                                                                                 'nutrition-related '
                                                                                                                                 'thread.'},
                                      'israel, right, israeli, arab, state, jews, attack, kill, country, land': {'label': 'Israel '
                                                                                                                          'and '
                                                                                                                          'the '
                                                                                                                          'Israeli–Palestinian '
                                                                                                                          'conflict',
                                                                                                                 'rationale': 'Israel, '
                                                                                                                              'Israeli, '
                                                                                                                              'Arab, '
                                                                                                                              'Jewish, '
                                                                                                                              'land, '
                                                                                                                              'state, '
                                                                                                                              'and '
                                                                                                                              'attack '
                                                                                                                              'indicate '
                                                                                                                              'discussion '
                                                                                                                              'of '
                                                                                                                              'Israel '
                                                                                                                              'and '
                                                                                                                              'the '
                                                                                                                              'surrounding '
                                                                                                                              'political '
                                                                                                                              'conflict.'},
                                      'home, copy, happen, cover, woman, start, live, hear, dog, black': {'label': 'Everyday '
                                                                                                                   'life '
                                                                                                                   'and '
                                                                                                                   'personal '
                                                                                                                   'stories '
                                                                                                                   '(weakly '
                                                                                                                   'focused)',
                                                                                                          'rationale': 'Home, '
                                                                                                                       'woman, '
                                                                                                                       'live, '
                                                                                                                       'hear, '
                                                                                                                       'and '
                                                                                                                       'dog '
                                                                                                                       'suggest '
                                                                                                                       'personal '
                                                                                                                       'or '
                                                                                                                       'domestic '
                                                                                                                       'narratives, '
                                                                                                                       'but '
                                                                                                                       'the '
                                                                                                                       'terms '
                                                                                                                       'do '
                                                                                                                       'not '
                                                                                                                       'form '
                                                                                                                       'a '
                                                                                                                       'sharply '
                                                                                                                       'defined '
                                                                                                                       'subject.'},
                                      'key, gun, law, chip, encryption, right, public, government, clipper, system': {'label': 'Encryption '
                                                                                                                               'policy '
                                                                                                                               'and '
                                                                                                                               'firearms '
                                                                                                                               'law '
                                                                                                                               '(mixed)',
                                                                                                                      'rationale': 'Key, '
                                                                                                                                   'chip, '
                                                                                                                                   'encryption, '
                                                                                                                                   'and '
                                                                                                                                   'Clipper '
                                                                                                                                   'point '
                                                                                                                                   'to '
                                                                                                                                   'cryptography '
                                                                                                                                   'policy; '
                                                                                                                                   'gun, '
                                                                                                                                   'law, '
                                                                                                                                   'and '
                                                                                                                                   'right '
                                                                                                                                   'introduce '
                                                                                                                                   'a '
                                                                                                                                   'separate '
                                                                                                                                   'firearms-policy '
                                                                                                                                   'theme.'},
                                      'window, program, file, display, line, run, set, entry, application, output': {'label': 'Windowed '
                                                                                                                              'software '
                                                                                                                              'and '
                                                                                                                              'program '
                                                                                                                              'use',
                                                                                                                     'rationale': 'Window, '
                                                                                                                                  'program, '
                                                                                                                                  'file, '
                                                                                                                                  'display, '
                                                                                                                                  'application, '
                                                                                                                                  'and '
                                                                                                                                  'output '
                                                                                                                                  'point '
                                                                                                                                  'to '
                                                                                                                                  'operating '
                                                                                                                                  'software '
                                                                                                                                  'and '
                                                                                                                                  'graphical '
                                                                                                                                  'user-interface '
                                                                                                                                  'discussions.'},
                                      'point, long, probably, group, read, idea, actually, course, little, bad': {'label': 'General '
                                                                                                                           'discussion '
                                                                                                                           'and '
                                                                                                                           'argument '
                                                                                                                           '(weakly '
                                                                                                                           'focused)',
                                                                                                                  'rationale': 'The '
                                                                                                                               'terms '
                                                                                                                               'are '
                                                                                                                               'mostly '
                                                                                                                               'conversational '
                                                                                                                               'rather '
                                                                                                                               'than '
                                                                                                                               'domain-specific, '
                                                                                                                               'suggesting '
                                                                                                                               'broad '
                                                                                                                               'discussion '
                                                                                                                               'or '
                                                                                                                               'argument '
                                                                                                                               'without '
                                                                                                                               'a '
                                                                                                                               'distinct '
                                                                                                                               'subject.'},
                                      'file, image, available, version, ftp, program, window, edu, software, graphic': {'label': 'Digital '
                                                                                                                                 'images '
                                                                                                                                 'and '
                                                                                                                                 'graphics '
                                                                                                                                 'software',
                                                                                                                        'rationale': 'Image, '
                                                                                                                                     'graphic, '
                                                                                                                                     'file, '
                                                                                                                                     'FTP, '
                                                                                                                                     'version, '
                                                                                                                                     'and '
                                                                                                                                     'software '
                                                                                                                                     'describe '
                                                                                                                                     'digital '
                                                                                                                                     'graphics '
                                                                                                                                     'and '
                                                                                                                                     'exchanging '
                                                                                                                                     'image '
                                                                                                                                     'files.'},
                                      'game, team, play, win, player, season, hockey, league, run, score': {'label': 'Team '
                                                                                                                     'sports, '
                                                                                                                     'especially '
                                                                                                                     'hockey',
                                                                                                            'rationale': 'Game, '
                                                                                                                         'team, '
                                                                                                                         'player, '
                                                                                                                         'season, '
                                                                                                                         'league, '
                                                                                                                         'and '
                                                                                                                         'score '
                                                                                                                         'are '
                                                                                                                         'sports '
                                                                                                                         'terms; '
                                                                                                                         'hockey '
                                                                                                                         'provides '
                                                                                                                         'the '
                                                                                                                         'clearest '
                                                                                                                         'specific '
                                                                                                                         'sport.'},
                                      'win, excellent, icon, mask, miss, condition, david, zip, cica, camera': {'label': 'Digital '
                                                                                                                         'photography '
                                                                                                                         'and '
                                                                                                                         'image '
                                                                                                                         'utilities '
                                                                                                                         '(weakly '
                                                                                                                         'focused)',
                                                                                                                'rationale': 'Icon, '
                                                                                                                             'mask, '
                                                                                                                             'ZIP, '
                                                                                                                             'CICA, '
                                                                                                                             'and '
                                                                                                                             'camera '
                                                                                                                             'suggest '
                                                                                                                             'image '
                                                                                                                             'or '
                                                                                                                             'software '
                                                                                                                             'utilities, '
                                                                                                                             'but '
                                                                                                                             'several '
                                                                                                                             'terms '
                                                                                                                             'are '
                                                                                                                             'ambiguous '
                                                                                                                             'and '
                                                                                                                             'the '
                                                                                                                             'topic '
                                                                                                                             'is '
                                                                                                                             'only '
                                                                                                                             'loosely '
                                                                                                                             'focused.'}},
 'NMF_BoW (Controlled)': {'file, available, version, ftp, image, edu, software, program, include, graphic': {'label': 'Software '
                                                                                                                      'files '
                                                                                                                      'and '
                                                                                                                      'graphics '
                                                                                                                      'exchange',
                                                                                                             'rationale': 'Files, '
                                                                                                                          'FTP, '
                                                                                                                          'versions, '
                                                                                                                          'image '
                                                                                                                          'formats, '
                                                                                                                          'and '
                                                                                                                          'graphics '
                                                                                                                          'describe '
                                                                                                                          'software '
                                                                                                                          'and '
                                                                                                                          'digital-file '
                                                                                                                          'sharing.'},
                          'gun, law, right, state, crime, weapon, firearm, file, control, police': {'label': 'Firearms, '
                                                                                                             'law, '
                                                                                                             'and '
                                                                                                             'gun '
                                                                                                             'policy',
                                                                                                    'rationale': 'Gun, '
                                                                                                                 'firearm, '
                                                                                                                 'weapon, '
                                                                                                                 'law, '
                                                                                                                 'crime, '
                                                                                                                 'and '
                                                                                                                 'police '
                                                                                                                 'terms '
                                                                                                                 'center '
                                                                                                                 'on '
                                                                                                                 'firearms '
                                                                                                                 'and '
                                                                                                                 'regulation.'},
                          'president, job, money, government, support, pay, program, talk, business, clinton': {'label': 'US '
                                                                                                                         'politics, '
                                                                                                                         'government, '
                                                                                                                         'and '
                                                                                                                         'economics '
                                                                                                                         '(mixed)',
                                                                                                                'rationale': 'Presidential '
                                                                                                                             'and '
                                                                                                                             'government '
                                                                                                                             'terms '
                                                                                                                             'combine '
                                                                                                                             'with '
                                                                                                                             'jobs, '
                                                                                                                             'money, '
                                                                                                                             'business, '
                                                                                                                             'and '
                                                                                                                             'support, '
                                                                                                                             'making '
                                                                                                                             'this '
                                                                                                                             'a '
                                                                                                                             'broad '
                                                                                                                             'public-affairs '
                                                                                                                             'topic.'},
                          'bad, right, car, little, long, maybe, big, probably, bike, let': {'label': 'General '
                                                                                                      'discussion '
                                                                                                      'with '
                                                                                                      'vehicle '
                                                                                                      'overlap '
                                                                                                      '(weakly '
                                                                                                      'focused)',
                                                                                             'rationale': 'Most '
                                                                                                          'terms '
                                                                                                          'are '
                                                                                                          'generic '
                                                                                                          'conversational '
                                                                                                          'language; '
                                                                                                          'car '
                                                                                                          'and '
                                                                                                          'bike '
                                                                                                          'appear '
                                                                                                          'but '
                                                                                                          'do '
                                                                                                          'not '
                                                                                                          'form '
                                                                                                          'a '
                                                                                                          'sufficiently '
                                                                                                          'distinctive '
                                                                                                          'subject.'},
                          'drive, disk, scsi, hard, system, controller, floppy, tape, ide, bio': {'label': 'Computer '
                                                                                                           'drives '
                                                                                                           'and '
                                                                                                           'storage '
                                                                                                           'hardware',
                                                                                                  'rationale': 'Drive, '
                                                                                                               'disk, '
                                                                                                               'SCSI, '
                                                                                                               'controller, '
                                                                                                               'floppy, '
                                                                                                               'tape, '
                                                                                                               'and '
                                                                                                               'IDE '
                                                                                                               'are '
                                                                                                               'specific '
                                                                                                               'to '
                                                                                                               'computer '
                                                                                                               'storage '
                                                                                                               'and '
                                                                                                               'interfaces.'},
                          'window, file, program, run, set, application, display, entry, widget, error': {'label': 'Window '
                                                                                                                   'systems '
                                                                                                                   'and '
                                                                                                                   'graphical '
                                                                                                                   'software',
                                                                                                          'rationale': 'Window, '
                                                                                                                       'application, '
                                                                                                                       'display, '
                                                                                                                       'widget, '
                                                                                                                       'and '
                                                                                                                       'program '
                                                                                                                       'identify '
                                                                                                                       'graphical-interface '
                                                                                                                       'software '
                                                                                                                       'and '
                                                                                                                       'troubleshooting.'},
                          'game, team, play, win, player, season, league, hockey, period, goal': {'label': 'Team '
                                                                                                           'sports, '
                                                                                                           'especially '
                                                                                                           'hockey '
                                                                                                           '(baseball '
                                                                                                           'overlap)',
                                                                                                  'rationale': 'Game, '
                                                                                                               'team, '
                                                                                                               'player, '
                                                                                                               'season, '
                                                                                                               'and '
                                                                                                               'league '
                                                                                                               'are '
                                                                                                               'shared '
                                                                                                               'sports '
                                                                                                               'terms; '
                                                                                                               'hockey, '
                                                                                                               'period, '
                                                                                                               'and '
                                                                                                               'goal '
                                                                                                               'point '
                                                                                                               'more '
                                                                                                               'strongly '
                                                                                                               'to '
                                                                                                               'hockey, '
                                                                                                               'while '
                                                                                                               'the '
                                                                                                               'model '
                                                                                                               'also '
                                                                                                               'groups '
                                                                                                               'baseball '
                                                                                                               'posts '
                                                                                                               'here.'},
                          'information, group, internet, news, article, computer, newsgroup, service, message, net': {'label': 'Internet '
                                                                                                                               'and '
                                                                                                                               'newsgroup '
                                                                                                                               'communication',
                                                                                                                      'rationale': 'Newsgroups, '
                                                                                                                                   'messages, '
                                                                                                                                   'articles, '
                                                                                                                                   'internet '
                                                                                                                                   'services, '
                                                                                                                                   'and '
                                                                                                                                   'networks '
                                                                                                                                   'describe '
                                                                                                                                   'online '
                                                                                                                                   'communication '
                                                                                                                                   'infrastructure.'},
                          'power, car, ground, wire, engine, light, water, circuit, current, need': {'label': 'Electrical '
                                                                                                              'systems '
                                                                                                              'and '
                                                                                                              'vehicle '
                                                                                                              'troubleshooting '
                                                                                                              '(mixed)',
                                                                                                     'rationale': 'Power, '
                                                                                                                  'wiring, '
                                                                                                                  'current, '
                                                                                                                  'and '
                                                                                                                  'circuits '
                                                                                                                  'form '
                                                                                                                  'an '
                                                                                                                  'electrical '
                                                                                                                  'theme, '
                                                                                                                  'with '
                                                                                                                  'car '
                                                                                                                  'and '
                                                                                                                  'engine '
                                                                                                                  'terms '
                                                                                                                  'showing '
                                                                                                                  'overlap '
                                                                                                                  'with '
                                                                                                                  'vehicle '
                                                                                                                  'discussions.'},
                          'space, launch, nasa, satellite, earth, orbit, mission, system, shuttle, technology': {'label': 'Spaceflight, '
                                                                                                                          'satellites, '
                                                                                                                          'and '
                                                                                                                          'NASA '
                                                                                                                          'missions',
                                                                                                                 'rationale': 'NASA, '
                                                                                                                              'launch, '
                                                                                                                              'satellite, '
                                                                                                                              'orbit, '
                                                                                                                              'mission, '
                                                                                                                              'shuttle, '
                                                                                                                              'and '
                                                                                                                              'Earth '
                                                                                                                              'give '
                                                                                                                              'this '
                                                                                                                              'topic '
                                                                                                                              'a '
                                                                                                                              'clear '
                                                                                                                              'space-exploration '
                                                                                                                              'focus.'},
                          'god, jesus, love, book, christ, life, bible, write, john, sin': {'label': 'Christianity, '
                                                                                                     'scripture, '
                                                                                                     'and '
                                                                                                     'religious '
                                                                                                     'life',
                                                                                            'rationale': 'God, '
                                                                                                         'Jesus, '
                                                                                                         'Christ, '
                                                                                                         'Bible, '
                                                                                                         'sin, '
                                                                                                         'and '
                                                                                                         'John '
                                                                                                         'identify '
                                                                                                         'Christian '
                                                                                                         'scripture '
                                                                                                         'and '
                                                                                                         'belief; '
                                                                                                         'love '
                                                                                                         'and '
                                                                                                         'life '
                                                                                                         'are '
                                                                                                         'common '
                                                                                                         'religious '
                                                                                                         'discussion '
                                                                                                         'terms.'},
                          'god, believe, claim, religion, exist, point, argument, true, church, christian': {'label': 'Arguments '
                                                                                                                      'about '
                                                                                                                      'belief '
                                                                                                                      'and '
                                                                                                                      'religion',
                                                                                                             'rationale': 'Belief, '
                                                                                                                          'existence, '
                                                                                                                          'claims, '
                                                                                                                          'arguments, '
                                                                                                                          'and '
                                                                                                                          'religion '
                                                                                                                          'indicate '
                                                                                                                          'debate '
                                                                                                                          'about '
                                                                                                                          'faith '
                                                                                                                          'and '
                                                                                                                          'the '
                                                                                                                          'existence '
                                                                                                                          'of '
                                                                                                                          'God.'},
                          'kill, happen, woman, start, home, child, fire, live, hear, armenian': {'label': 'Violence '
                                                                                                           'and '
                                                                                                           'personal '
                                                                                                           'events '
                                                                                                           '(mixed)',
                                                                                                  'rationale': 'Kill '
                                                                                                               'and '
                                                                                                               'fire '
                                                                                                               'suggest '
                                                                                                               'violent '
                                                                                                               'events, '
                                                                                                               'while '
                                                                                                               'woman, '
                                                                                                               'child, '
                                                                                                               'home, '
                                                                                                               'and '
                                                                                                               'live '
                                                                                                               'suggest '
                                                                                                               'personal '
                                                                                                               'reports. '
                                                                                                               'Armenian '
                                                                                                               'adds '
                                                                                                               'a '
                                                                                                               'possible '
                                                                                                               'historical-conflict '
                                                                                                               'thread, '
                                                                                                               'so '
                                                                                                               'this '
                                                                                                               'topic '
                                                                                                               'is '
                                                                                                               'mixed.'},
                          'sell, offer, price, max, black, sale, cover, soon, include, book': {'label': 'Classified '
                                                                                                        'listings '
                                                                                                        'and '
                                                                                                        'items '
                                                                                                        'for '
                                                                                                        'sale',
                                                                                               'rationale': 'Sell, '
                                                                                                            'offer, '
                                                                                                            'price, '
                                                                                                            'and '
                                                                                                            'sale '
                                                                                                            'are '
                                                                                                            'characteristic '
                                                                                                            'of '
                                                                                                            'marketplace '
                                                                                                            'posts; '
                                                                                                            'book '
                                                                                                            'and '
                                                                                                            'cover '
                                                                                                            'suggest '
                                                                                                            'items '
                                                                                                            'being '
                                                                                                            'listed.'},
                          'card, mac, bit, monitor, driver, video, memory, machine, apple, chip': {'label': 'Computer '
                                                                                                            'hardware '
                                                                                                            'and '
                                                                                                            'display '
                                                                                                            'devices',
                                                                                                   'rationale': 'Card, '
                                                                                                                'monitor, '
                                                                                                                'video '
                                                                                                                'driver, '
                                                                                                                'memory, '
                                                                                                                'machine, '
                                                                                                                'Apple, '
                                                                                                                'and '
                                                                                                                'chip '
                                                                                                                'identify '
                                                                                                                'computer '
                                                                                                                'hardware '
                                                                                                                'and '
                                                                                                                'graphics '
                                                                                                                'equipment.'},
                          'datum, line, model, image, point, bit, level, value, object, code': {'label': 'Computer '
                                                                                                         'graphics '
                                                                                                         'and '
                                                                                                         'data '
                                                                                                         'representation '
                                                                                                         '(broad)',
                                                                                                'rationale': 'Image, '
                                                                                                             'model, '
                                                                                                             'object, '
                                                                                                             'point, '
                                                                                                             'and '
                                                                                                             'value '
                                                                                                             'suggest '
                                                                                                             'graphics '
                                                                                                             'or '
                                                                                                             'technical '
                                                                                                             'data '
                                                                                                             'representation, '
                                                                                                             'though '
                                                                                                             'the '
                                                                                                             'terms '
                                                                                                             'are '
                                                                                                             'broad '
                                                                                                             'and '
                                                                                                             'the '
                                                                                                             'topic '
                                                                                                             'is '
                                                                                                             'not '
                                                                                                             'tightly '
                                                                                                             'focused.'},
                          'israel, war, israeli, arab, jews, state, right, land, attack, country': {'label': 'Israel '
                                                                                                             'and '
                                                                                                             'Middle '
                                                                                                             'East '
                                                                                                             'political '
                                                                                                             'conflict',
                                                                                                    'rationale': 'Israel, '
                                                                                                                 'Israeli, '
                                                                                                                 'Arab, '
                                                                                                                 'Jewish, '
                                                                                                                 'land, '
                                                                                                                 'war, '
                                                                                                                 'and '
                                                                                                                 'attack '
                                                                                                                 'identify '
                                                                                                                 'discussion '
                                                                                                                 'of '
                                                                                                                 'Israel '
                                                                                                                 'and '
                                                                                                                 'Middle '
                                                                                                                 'East '
                                                                                                                 'conflict.'},
                          'study, drug, cause, disease, health, medical, food, patient, doctor, report': {'label': 'Medicine, '
                                                                                                                   'disease, '
                                                                                                                   'and '
                                                                                                                   'health '
                                                                                                                   'research',
                                                                                                          'rationale': 'Study, '
                                                                                                                       'disease, '
                                                                                                                       'health, '
                                                                                                                       'medical, '
                                                                                                                       'patient, '
                                                                                                                       'and '
                                                                                                                       'doctor '
                                                                                                                       'indicate '
                                                                                                                       'medical '
                                                                                                                       'research '
                                                                                                                       'and '
                                                                                                                       'health-related '
                                                                                                                       'discussion.'},
                          'key, chip, encryption, clipper, government, system, security, phone, bit, public': {'label': 'Cryptography, '
                                                                                                                        'computer '
                                                                                                                        'security, '
                                                                                                                        'and '
                                                                                                                        'privacy '
                                                                                                                        'policy',
                                                                                                               'rationale': 'Key, '
                                                                                                                            'encryption, '
                                                                                                                            'Clipper, '
                                                                                                                            'security, '
                                                                                                                            'phone, '
                                                                                                                            'and '
                                                                                                                            'government '
                                                                                                                            'point '
                                                                                                                            'to '
                                                                                                                            'cryptography '
                                                                                                                            'and '
                                                                                                                            'debates '
                                                                                                                            'about '
                                                                                                                            'secure '
                                                                                                                            'communications.'},
                          'armenian, turkish, muslim, nazi, greek, turkey, turk, history, government, genocide': {'label': 'Armenian, '
                                                                                                                           'Turkish, '
                                                                                                                           'and '
                                                                                                                           'regional '
                                                                                                                           'history',
                                                                                                                  'rationale': 'Armenian, '
                                                                                                                               'Turkish, '
                                                                                                                               'Greek, '
                                                                                                                               'Turkey, '
                                                                                                                               'history, '
                                                                                                                               'and '
                                                                                                                               'genocide '
                                                                                                                               'identify '
                                                                                                                               'historical '
                                                                                                                               'and '
                                                                                                                               'political '
                                                                                                                               'discussion '
                                                                                                                               'of '
                                                                                                                               'ethnic '
                                                                                                                               'conflict '
                                                                                                                               'in '
                                                                                                                               'the '
                                                                                                                               'region.'}},
 'NMF_TFIDF (Practical)': {'point, reason, case, claim, argument, mind, fact, talk, person, wrong': {'label': 'Reasoning '
                                                                                                              'and '
                                                                                                              'argumentation',
                                                                                                     'rationale': 'Reason, '
                                                                                                                  'claim, '
                                                                                                                  'argument, '
                                                                                                                  'fact, '
                                                                                                                  'and '
                                                                                                                  'wrong '
                                                                                                                  'describe '
                                                                                                                  'discussion '
                                                                                                                  'centered '
                                                                                                                  'on '
                                                                                                                  'opinions '
                                                                                                                  'and '
                                                                                                                  'evidence '
                                                                                                                  'rather '
                                                                                                                  'than '
                                                                                                                  'a '
                                                                                                                  'narrow '
                                                                                                                  'domain.'},
                           'window, program, run, application, server, software, display, user, code, motif': {'label': 'Window '
                                                                                                                        'software '
                                                                                                                        'and '
                                                                                                                        'application '
                                                                                                                        'development',
                                                                                                               'rationale': 'Window '
                                                                                                                            'systems, '
                                                                                                                            'applications, '
                                                                                                                            'servers, '
                                                                                                                            'display, '
                                                                                                                            'code, '
                                                                                                                            'and '
                                                                                                                            'Motif '
                                                                                                                            'identify '
                                                                                                                            'software '
                                                                                                                            'and '
                                                                                                                            'graphical-interface '
                                                                                                                            'development.'},
                           'game, team, play, player, win, season, fan, hockey, league, baseball': {'label': 'Team '
                                                                                                             'sports '
                                                                                                             '(baseball '
                                                                                                             'and '
                                                                                                             'hockey '
                                                                                                             'grouped '
                                                                                                             'together)',
                                                                                                    'rationale': 'Game, '
                                                                                                                 'team, '
                                                                                                                 'player, '
                                                                                                                 'season, '
                                                                                                                 'and '
                                                                                                                 'league '
                                                                                                                 'are '
                                                                                                                 'common '
                                                                                                                 'sports '
                                                                                                                 'language; '
                                                                                                                 'baseball '
                                                                                                                 'and '
                                                                                                                 'hockey '
                                                                                                                 'both '
                                                                                                                 'appear, '
                                                                                                                 'so '
                                                                                                                 'this '
                                                                                                                 'model '
                                                                                                                 'does '
                                                                                                                 'not '
                                                                                                                 'fully '
                                                                                                                 'distinguish '
                                                                                                                 'the '
                                                                                                                 'two '
                                                                                                                 'sports.'},
                           'drive, disk, hard, scsi, controller, floppy, ide, hard drive, rom, boot': {'label': 'Computer '
                                                                                                                'storage '
                                                                                                                'and '
                                                                                                                'drive '
                                                                                                                'hardware',
                                                                                                       'rationale': 'Hard '
                                                                                                                    'drives, '
                                                                                                                    'disks, '
                                                                                                                    'SCSI, '
                                                                                                                    'IDE, '
                                                                                                                    'controllers, '
                                                                                                                    'and '
                                                                                                                    'boot '
                                                                                                                    'media '
                                                                                                                    'identify '
                                                                                                                    'storage '
                                                                                                                    'hardware '
                                                                                                                    'and '
                                                                                                                    'configuration.'},
                           'key, chip, encryption, clipper, government, algorithm, phone, public, nsa, system': {'label': 'Cryptography, '
                                                                                                                          'encryption, '
                                                                                                                          'and '
                                                                                                                          'privacy '
                                                                                                                          'policy',
                                                                                                                 'rationale': 'Encryption, '
                                                                                                                              'keys, '
                                                                                                                              'Clipper, '
                                                                                                                              'algorithms, '
                                                                                                                              'NSA, '
                                                                                                                              'and '
                                                                                                                              'public '
                                                                                                                              'systems '
                                                                                                                              'indicate '
                                                                                                                              'cryptography '
                                                                                                                              'and '
                                                                                                                              'privacy '
                                                                                                                              'debates.'},
                           'soon, food, bank, doctor, disease, pain, msg, patient, brain, eat': {'label': 'Health, '
                                                                                                          'symptoms, '
                                                                                                          'and '
                                                                                                          'medical '
                                                                                                          'advice',
                                                                                                 'rationale': 'Doctors, '
                                                                                                              'disease, '
                                                                                                              'pain, '
                                                                                                              'patients, '
                                                                                                              'brain, '
                                                                                                              'and '
                                                                                                              'food '
                                                                                                              'point '
                                                                                                              'to '
                                                                                                              'medical '
                                                                                                              'and '
                                                                                                              'health '
                                                                                                              'discussions; '
                                                                                                              'a '
                                                                                                              'few '
                                                                                                              'terms '
                                                                                                              'are '
                                                                                                              'unrelated '
                                                                                                              'noise.'},
                           'god, jesus, christian, believe, bible, religion, church, belief, faith, christ': {'label': 'Christian '
                                                                                                                       'belief '
                                                                                                                       'and '
                                                                                                                       'religious '
                                                                                                                       'practice',
                                                                                                              'rationale': 'God, '
                                                                                                                           'Jesus, '
                                                                                                                           'Christian, '
                                                                                                                           'Bible, '
                                                                                                                           'church, '
                                                                                                                           'faith, '
                                                                                                                           'and '
                                                                                                                           'Christ '
                                                                                                                           'form '
                                                                                                                           'a '
                                                                                                                           'coherent '
                                                                                                                           'religious '
                                                                                                                           'topic.'},
                           'card, monitor, driver, video, board, apple, mode, color, ram, buy': {'label': 'Computer '
                                                                                                          'graphics '
                                                                                                          'cards '
                                                                                                          'and '
                                                                                                          'display '
                                                                                                          'hardware',
                                                                                                 'rationale': 'Cards, '
                                                                                                              'monitors, '
                                                                                                              'video '
                                                                                                              'drivers, '
                                                                                                              'boards, '
                                                                                                              'display '
                                                                                                              'modes, '
                                                                                                              'color, '
                                                                                                              'and '
                                                                                                              'RAM '
                                                                                                              'describe '
                                                                                                              'computer '
                                                                                                              'hardware.'},
                           'israel, kill, law, right, state, government, gun, country, weapon, war': {'label': 'Politics, '
                                                                                                               'conflict, '
                                                                                                               'and '
                                                                                                               'firearms '
                                                                                                               '(mixed)',
                                                                                                      'rationale': 'Israel, '
                                                                                                                   'war, '
                                                                                                                   'and '
                                                                                                                   'government '
                                                                                                                   'combine '
                                                                                                                   'with '
                                                                                                                   'gun, '
                                                                                                                   'weapon, '
                                                                                                                   'and '
                                                                                                                   'law '
                                                                                                                   'vocabulary, '
                                                                                                                   'so '
                                                                                                                   'this '
                                                                                                                   'topic '
                                                                                                                   'spans '
                                                                                                                   'political '
                                                                                                                   'conflict '
                                                                                                                   'and '
                                                                                                                   'firearms.'},
                           'car, engine, mile, yeah, owner, road, dealer, auto, oil, drive': {'label': 'Automobiles, '
                                                                                                       'ownership, '
                                                                                                       'and '
                                                                                                       'maintenance',
                                                                                              'rationale': 'Car, '
                                                                                                           'engine, '
                                                                                                           'mileage, '
                                                                                                           'owner, '
                                                                                                           'road, '
                                                                                                           'dealer, '
                                                                                                           'oil, '
                                                                                                           'and '
                                                                                                           'drive '
                                                                                                           'point '
                                                                                                           'to '
                                                                                                           'vehicle '
                                                                                                           'ownership '
                                                                                                           'and '
                                                                                                           'maintenance.'},
                           'file, ftp, site, picture, format, edu, directory, faq, convert, program': {'label': 'Digital '
                                                                                                                'files, '
                                                                                                                'image '
                                                                                                                'formats, '
                                                                                                                'and '
                                                                                                                'FTP '
                                                                                                                'sites',
                                                                                                       'rationale': 'File, '
                                                                                                                    'FTP, '
                                                                                                                    'picture, '
                                                                                                                    'format, '
                                                                                                                    'directory, '
                                                                                                                    'and '
                                                                                                                    'convert '
                                                                                                                    'describe '
                                                                                                                    'sharing '
                                                                                                                    'and '
                                                                                                                    'converting '
                                                                                                                    'digital '
                                                                                                                    'files, '
                                                                                                                    'especially '
                                                                                                                    'images.'},
                           'sell, price, offer, sale, interested, pay, condition, include, buy, box': {'label': 'Marketplace '
                                                                                                                'listings '
                                                                                                                'and '
                                                                                                                'sales',
                                                                                                       'rationale': 'Sell, '
                                                                                                                    'price, '
                                                                                                                    'offer, '
                                                                                                                    'sale, '
                                                                                                                    'interested, '
                                                                                                                    'and '
                                                                                                                    'buy '
                                                                                                                    'indicate '
                                                                                                                    'classified '
                                                                                                                    'advertisements '
                                                                                                                    'and '
                                                                                                                    'transactions.'},
                           'armenian, woman, greek, turkish, burn, dead, plane, turkey, german, body': {'label': 'Ethnic '
                                                                                                                 'conflict '
                                                                                                                 'and '
                                                                                                                 'violent '
                                                                                                                 'events '
                                                                                                                 '(mixed)',
                                                                                                        'rationale': 'Armenian, '
                                                                                                                     'Greek, '
                                                                                                                     'Turkish, '
                                                                                                                     'and '
                                                                                                                     'Turkey '
                                                                                                                     'indicate '
                                                                                                                     'regional '
                                                                                                                     'ethnic '
                                                                                                                     'conflict; '
                                                                                                                     'burn, '
                                                                                                                     'dead, '
                                                                                                                     'plane, '
                                                                                                                     'and '
                                                                                                                     'body '
                                                                                                                     'suggest '
                                                                                                                     'reports '
                                                                                                                     'of '
                                                                                                                     'violent '
                                                                                                                     'events.'},
                           'bike, stuff, ride, delete, head, dod, motorcycle, little, turn, right': {'label': 'Motorcycles '
                                                                                                              'and '
                                                                                                              'riding',
                                                                                                     'rationale': 'Bike, '
                                                                                                                  'ride, '
                                                                                                                  'and '
                                                                                                                  'motorcycle '
                                                                                                                  'are '
                                                                                                                  'the '
                                                                                                                  'strongest '
                                                                                                                  'terms, '
                                                                                                                  'with '
                                                                                                                  'additional '
                                                                                                                  'broad '
                                                                                                                  'conversational '
                                                                                                                  'language.'},
                           'information, university, internet, school, fax, service, april, date, research, computer': {'label': 'University '
                                                                                                                                 'and '
                                                                                                                                 'Internet '
                                                                                                                                 'information '
                                                                                                                                 'services',
                                                                                                                        'rationale': 'University, '
                                                                                                                                     'Internet, '
                                                                                                                                     'service, '
                                                                                                                                     'research, '
                                                                                                                                     'and '
                                                                                                                                     'computer '
                                                                                                                                     'point '
                                                                                                                                     'to '
                                                                                                                                     'academic '
                                                                                                                                     'and '
                                                                                                                                     'online '
                                                                                                                                     'information '
                                                                                                                                     'resources; '
                                                                                                                                     'date '
                                                                                                                                     'and '
                                                                                                                                     'fax '
                                                                                                                                     'suggest '
                                                                                                                                     'directory-style '
                                                                                                                                     'posts.'},
                           'speed, fast, rate, mhz, gun, system, slow, memory, run, datum': {'label': 'Computer '
                                                                                                      'performance '
                                                                                                      'and '
                                                                                                      'hardware '
                                                                                                      'speed '
                                                                                                      '(mixed)',
                                                                                             'rationale': 'Speed, '
                                                                                                          'rate, '
                                                                                                          'MHz, '
                                                                                                          'memory, '
                                                                                                          'and '
                                                                                                          'system '
                                                                                                          'suggest '
                                                                                                          'computer '
                                                                                                          'performance; '
                                                                                                          'gun '
                                                                                                          'is '
                                                                                                          'an '
                                                                                                          'unrelated '
                                                                                                          'term '
                                                                                                          'that '
                                                                                                          'makes '
                                                                                                          'this '
                                                                                                          'topic '
                                                                                                          'mixed.'},
                           'space, money, cost, air, launch, spend, nasa, water, earth, station': {'label': 'Space '
                                                                                                            'missions, '
                                                                                                            'NASA, '
                                                                                                            'and '
                                                                                                            'project '
                                                                                                            'costs',
                                                                                                   'rationale': 'Space, '
                                                                                                                'launch, '
                                                                                                                'NASA, '
                                                                                                                'Earth, '
                                                                                                                'and '
                                                                                                                'station '
                                                                                                                'identify '
                                                                                                                'space '
                                                                                                                'exploration; '
                                                                                                                'money, '
                                                                                                                'cost, '
                                                                                                                'and '
                                                                                                                'spend '
                                                                                                                'suggest '
                                                                                                                'mission '
                                                                                                                'funding.'},
                           'hear, read, article, group, answer, number, modem, week, ago, response': {'label': 'Newsgroup '
                                                                                                               'discussion '
                                                                                                               'and '
                                                                                                               'replies',
                                                                                                      'rationale': 'Article, '
                                                                                                                   'group, '
                                                                                                                   'answer, '
                                                                                                                   'and '
                                                                                                                   'response '
                                                                                                                   'indicate '
                                                                                                                   'online '
                                                                                                                   'discussion '
                                                                                                                   'threads; '
                                                                                                                   'modem '
                                                                                                                   'and '
                                                                                                                   'week '
                                                                                                                   'appear '
                                                                                                                   'as '
                                                                                                                   'supporting '
                                                                                                                   'context.'},
                           'power, bill, current, line, supply, need, job, signal, ground, output': {'label': 'Electrical '
                                                                                                              'power '
                                                                                                              'and '
                                                                                                              'utility '
                                                                                                              'systems',
                                                                                                     'rationale': 'Power, '
                                                                                                                  'current, '
                                                                                                                  'line, '
                                                                                                                  'supply, '
                                                                                                                  'signal, '
                                                                                                                  'ground, '
                                                                                                                  'and '
                                                                                                                  'output '
                                                                                                                  'are '
                                                                                                                  'electrical '
                                                                                                                  'engineering '
                                                                                                                  'and '
                                                                                                                  'utility '
                                                                                                                  'terms.'},
                           'value, printer, font, image, print, define, text, character, object, objective': {'label': 'Typography, '
                                                                                                                       'fonts, '
                                                                                                                       'and '
                                                                                                                       'printing',
                                                                                                              'rationale': 'Printer, '
                                                                                                                           'font, '
                                                                                                                           'print, '
                                                                                                                           'text, '
                                                                                                                           'and '
                                                                                                                           'character '
                                                                                                                           'identify '
                                                                                                                           'typography '
                                                                                                                           'and '
                                                                                                                           'computer '
                                                                                                                           'printing; '
                                                                                                                           'image '
                                                                                                                           'and '
                                                                                                                           'object '
                                                                                                                           'add '
                                                                                                                           'graphics '
                                                                                                                           'context.'}}}


from pathlib import Path

BASE_DIR = PROJECT_ROOT
DATA_DIR = BASE_DIR / "data"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TARGET_DOC_COUNT = 55_000
MIN_DOC_CHAR_LEN = 40
MIN_CORPUS_DOCS = 50_000
MIN_EXPLORATORY_TRAIN_DOCS = 1_000
MIN_DF = 20
MAX_DF = 0.35
MAX_FEATURES = 15_000
NGRAM_RANGE = (1, 2)
VECTORIZER_STOPWORDS = {
    "thank", "advance", "appreciate", "anybody", "please", "reply", "sorry",
    "info", "help", "mail", "email", "address", "send", "list",
}
# The selected corpus has 20 distinct newsgroups; matching this count gives
# the unsupervised models room to separate its finer-grained subjects.
N_TOPICS = 20
MAX_ITER_LDA = 40
MAX_ITER_NMF = 400
TOP_N_WORDS = 10
print(f"Project folder: {PROJECT_ROOT}")
print(f"Dataset source: {DATA_SOURCE}")


Project folder: c:\Users\elhat\Desktop\NLP_Project
Dataset source: 20newsgroups


## Step 1 — Acquire the corpus

Download or load the selected source, preserve its train/test split, and write unlabeled text plus dataset metadata. Evaluation labels are stored separately. Re-running replaces the current raw corpus and label file.

In [2]:
# Step 1 implementation (step1_acquire.py)
"""Acquire an unlabeled corpus and write provenance metadata."""

import argparse
import json
from datetime import datetime, timezone

import datasets
import pandas as pd
import sklearn
from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups, load_files
from sklearn.datasets._twenty_newsgroups import strip_newsgroup_footer, strip_newsgroup_header, strip_newsgroup_quoting



def acquire_ag_news() -> tuple[pd.DataFrame, dict]:
    """Load AG News with its official train/test split and isolate the labels."""
    print("Downloading AG News through Hugging Face Datasets...")
    dataset = load_dataset("fancyzhx/ag_news")
    split_frames = []
    for split in ("train", "test"):
        split_frame = pd.DataFrame(dataset[split])
        split_frame["split"] = split
        split_frames.append(split_frame)
    frame = pd.concat(split_frames, ignore_index=True)
    frame["text"] = frame["text"].astype(str).str.strip()
    frame = frame.loc[frame["text"].str.len() >= MIN_DOC_CHAR_LEN].reset_index(drop=True)
    frame.insert(0, "doc_id", range(len(frame)))
    labels_path = DATA_DIR / "isolated_labels_for_evaluation.parquet"
    frame[["doc_id", "label", "split"]].to_parquet(labels_path, index=False)
    unlabeled = frame[["doc_id", "text", "split"]]
    metadata = {
        "source": "AG News (fancyzhx/ag_news)",
        "dataset_id": "fancyzhx/ag_news",
        "acquired_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_library_version": datasets.__version__,
        "split_counts_before_filtering": {split: len(dataset[split]) for split in ("train", "test")},
        "split_counts_after_filtering": {str(k): int(v) for k, v in frame["split"].value_counts().items()},
        "sampling": "Official train/test splits retained; no sampling. Test documents and labels are held out from fitting.",
        "label_names": dataset["train"].features["label"].names,
        "total_documents_collected": len(unlabeled),
        "avg_character_length": round(float(unlabeled["text"].str.len().mean()), 2),
        "structure": "Unstructured news title and lead paragraph strings",
        "labels": f"Stored separately at {labels_path.name}; test labels are used only for external evaluation, never for preprocessing or model fitting.",
        "limitations": "Short news snippets can produce sparse word co-occurrence. The source has four broad editorial labels, so ten-topic models may identify subthemes.",
    }
    return unlabeled, metadata


def acquire_20newsgroups_chunked(n_docs: int) -> tuple[pd.DataFrame, dict]:
    """Split cleaned 20 Newsgroups posts into paragraph-level text units."""
    print("Fetching 20 Newsgroups through scikit-learn...")
    raw = fetch_20newsgroups(
        subset="all",
        remove=("headers", "footers", "quotes"),
        data_home=DATA_DIR / "sklearn_cache",
    )
    paragraphs = [part.strip() for doc in raw.data for part in doc.split("\n\n") if len(part.strip()) >= MIN_DOC_CHAR_LEN]
    frame = pd.DataFrame({"text": paragraphs}).drop_duplicates().reset_index(drop=True)
    eligible_paragraph_count = len(frame)
    if eligible_paragraph_count < n_docs:
        raise ValueError(
            f"20 Newsgroups produced only {eligible_paragraph_count:,} unique eligible paragraphs; {n_docs:,} requested. "
            "The official 20 Newsgroups corpus is the selected source for this project."
        )
    frame = frame.sample(n=n_docs, random_state=RANDOM_SEED).reset_index(drop=True)
    frame["split"] = "train"
    test_count = max(1, int(round(n_docs * 0.05)))
    test_ids = frame.sample(n=test_count, random_state=RANDOM_SEED).index
    frame.loc[test_ids, "split"] = "test"
    frame.insert(0, "doc_id", range(len(frame)))
    metadata = {
        "source": "20 Newsgroups, paragraph-chunked",
        "acquired_at_utc": datetime.now(timezone.utc).isoformat(),
        "sampling_seed": RANDOM_SEED,
        "requested_document_count": n_docs,
        "sampling": "Unique eligible paragraphs sampled without replacement; 5% random held-out test split.",
        "split_counts": {str(k): int(v) for k, v in frame["split"].value_counts().items()},
        "unique_eligible_paragraphs_before_sampling": eligible_paragraph_count,
        "total_documents_collected": len(frame),
        "avg_character_length": round(float(frame["text"].str.len().mean()), 2),
        "structure": "Paragraphs from posts with headers, footers, and quoted text removed",
        "limitations": "Paragraphs from one post are not independent documents; acquisition fails unless enough unique eligible chunks exist.",
    }
    return frame[["doc_id", "text", "split"]], metadata


def acquire_20newsgroups() -> tuple[pd.DataFrame, dict]:
    """Acquire the official 20 Newsgroups posts and keep its train/test split."""
    print("Fetching the official 20 Newsgroups train and test sets...")
    data_home = DATA_DIR / "sklearn_cache"
    remove = ("headers", "footers", "quotes")
    extracted_home = data_home / "20news_home"
    train_dir = extracted_home / "20news-bydate-train"
    test_dir = extracted_home / "20news-bydate-test"
    if train_dir.is_dir() and test_dir.is_dir():
        # sklearn's downloader extracts the official archive here before it
        # builds its pickle cache. Reuse complete extracted splits if a fetch
        # was interrupted after extraction.
        split_datasets = {
            "train": load_files(train_dir, encoding="latin1"),
            "test": load_files(test_dir, encoding="latin1"),
        }
    else:
        split_datasets = {
            "train": fetch_20newsgroups(subset="train", remove=remove, data_home=data_home),
            "test": fetch_20newsgroups(subset="test", remove=remove, data_home=data_home),
        }
    records = []
    split_counts_before = {}
    for split_name, corpus in split_datasets.items():
        split_counts_before[split_name] = len(corpus.data)
        texts = corpus.data
        texts = [strip_newsgroup_header(text) for text in texts]
        texts = [strip_newsgroup_footer(text) for text in texts]
        texts = [strip_newsgroup_quoting(text) for text in texts]
        part = pd.DataFrame({"text": texts, "label": corpus.target})
        part["split"] = split_name
        part["text"] = part["text"].astype(str).str.strip()
        part = part.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
        records.append(part)
    frame = pd.concat(records, ignore_index=True)
    frame = frame.loc[frame["text"].str.len() >= MIN_DOC_CHAR_LEN].reset_index(drop=True)
    frame.insert(0, "doc_id", range(len(frame)))
    labels_path = DATA_DIR / "isolated_labels_for_evaluation.parquet"
    frame[["doc_id", "label", "split"]].to_parquet(labels_path, index=False)
    unlabeled = frame[["doc_id", "text", "split"]]
    split_counts_after = {str(k): int(v) for k, v in frame["split"].value_counts().items()}
    return unlabeled, {
        "source": "20 Newsgroups (official post-level train/test split)",
        "dataset_id": "sklearn/20newsgroups",
        "acquired_at_utc": datetime.now(timezone.utc).isoformat(),
        "scikit_learn_version": sklearn.__version__,
        "sampling": "Official train/test posts; headers, footers, and quoted text removed. One post is one document.",
        "split_counts_before_filtering": split_counts_before,
        "split_counts_after_filtering": split_counts_after,
        "total_documents_collected": len(unlabeled),
        "minimum_documents_required": MIN_CORPUS_DOCS,
        "requirement_met": len(unlabeled) >= MIN_CORPUS_DOCS,
        "exploratory_under_minimum": len(unlabeled) < MIN_CORPUS_DOCS,
        "label_names": split_datasets["train"].target_names,
        "avg_character_length": round(float(unlabeled["text"].str.len().mean()), 2),
        "structure": "Unstructured newsgroup posts with headers, footers, and quoted passages removed",
        "labels": f"Stored separately at {labels_path.name}; test labels are used only for external evaluation, never preprocessing or model fitting.",
        "limitations": "The official corpus contains 20 newsgroups; some posts may remain closely related or share author/thread context.",
    }


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--source", choices=("ag_news", "20newsgroups", "20newsgroups_chunked"), default="20newsgroups")
    parser.add_argument("--n-docs", type=int, default=TARGET_DOC_COUNT,
                        help="Number of documents to sample (only used for 20newsgroups_chunked; ag_news always loads the full corpus).")
    args = parser.parse_args()
    if args.source == "ag_news":
        frame, metadata = acquire_ag_news()
    elif args.source == "20newsgroups":
        frame, metadata = acquire_20newsgroups()
    else:
        if args.n_docs < MIN_CORPUS_DOCS:
            parser.error(f"--n-docs must be at least {MIN_CORPUS_DOCS:,} for a compliant corpus.")
        frame, metadata = acquire_20newsgroups_chunked(args.n_docs)
    if len(frame) < MIN_CORPUS_DOCS and not metadata.get("exploratory_under_minimum", False):
        raise ValueError(f"Corpus has {len(frame):,} rows, below the {MIN_CORPUS_DOCS:,} document minimum.")
    if len(frame) < MIN_EXPLORATORY_TRAIN_DOCS:
        raise ValueError(f"Corpus has only {len(frame):,} rows; too small even for a useful exploratory fit.")
    frame.to_parquet(DATA_DIR / "raw_corpus.parquet", index=False)
    (DATA_DIR / "dataset_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Saved {len(frame):,} unlabeled documents to {DATA_DIR / 'raw_corpus.parquet'}")
    if len(frame) < MIN_CORPUS_DOCS:
        print("The selected 20 Newsgroups corpus is ready for this project run.")


sys.argv = ["step1_acquire", "--source", DATA_SOURCE]
main()


c:\Users\elhat\Desktop\NLP_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching the official 20 Newsgroups train and test sets...
Saved 18,028 unlabeled documents to c:\Users\elhat\Desktop\NLP_Project\data\raw_corpus.parquet
The selected 20 Newsgroups corpus is ready for this project run.


## Step 2 — Clean and preprocess

Decode HTML, normalize text, remove URLs/numbers, lemmatize with spaCy, filter stopwords, and drop very short or repetitive documents. This is the slowest step because spaCy processes every document. If interrupted, rerun this cell from the beginning. Install the model once if needed: `python -m spacy download en_core_web_sm`.

In [3]:
# Step 2 implementation (step2_preprocess.py)
"""Normalize, tokenize, lemmatize, and filter the corpus with spaCy."""

import argparse
from collections import Counter
import html
import json
import re

import pandas as pd
import spacy
from tqdm import tqdm


try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError as exc:
    raise RuntimeError("spaCy model missing. Run: python -m spacy download en_core_web_sm") from exc

HTML_TAG_RE = re.compile(r"<[^>]+>")
URL_EMAIL_NUMBER_RE = re.compile(r"(?:https?://\S+|www\.\S+|\S+@\S+|\b\d+\b)")
CUSTOM_STOPWORDS = {
    "say", "use", "make", "get", "go", "know", "take", "see", "come", "think", "look",
    "want", "give", "find", "tell", "ask", "work", "seem", "feel", "try", "leave", "call",
    "would", "could", "should", "also", "one", "two", "new", "time", "year", "day", "people",
    "way", "thing", "man", "world", "reuters", "ap", "afp", "quot", "gt", "lt", "href",
    "quickinfo", "fullquote", "don", "doesn", "didn", "isn", "wasn", "aren",
    "weren", "hasn", "haven", "hadn", "couldn", "wouldn", "shouldn",
    "like", "good", "problem", "question", "mean", "sure", "lot", "post",
}
FILTERED_STOPWORDS = nlp.Defaults.stop_words | CUSTOM_STOPWORDS
for word in CUSTOM_STOPWORDS:
    nlp.vocab[word].is_stop = True


def clean_raw_string(text: str) -> str:
    text = html.unescape(str(text)).lower()
    text = HTML_TAG_RE.sub(" ", text)
    text = URL_EMAIL_NUMBER_RE.sub(" ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _content_lemmas(doc) -> list[str]:
    """Keep alphabetic lemmas that are neither spaCy nor project stopwords."""
    lemmas = []
    for token in doc:
        lemma = token.lemma_.lower()
        if token.is_alpha and len(lemma) >= 3 and not token.is_stop and lemma not in FILTERED_STOPWORDS:
            lemmas.append(lemma)
    return lemmas


def preprocess_single_document(text: str) -> list[str]:
    """Apply training-equivalent preprocessing for inference."""
    doc = nlp(clean_raw_string(text))
    return _content_lemmas(doc)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--n-process", type=int, default=1, help="spaCy worker processes (default: 1 for portable, memory-safe execution).")
    args = parser.parse_args()
    if args.n_process < 1:
        parser.error("--n-process must be at least 1.")
    raw_path = DATA_DIR / "raw_corpus.parquet"
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw corpus not found: {raw_path}. Run src.step1_acquire first.")
    frame = pd.read_parquet(raw_path)
    metadata_path = DATA_DIR / "dataset_metadata.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {}
    exploratory = bool(metadata.get("exploratory_under_minimum", False))
    minimum_docs = MIN_EXPLORATORY_TRAIN_DOCS if exploratory else MIN_CORPUS_DOCS
    if not {"doc_id", "text", "split"}.issubset(frame.columns) or frame.empty:
        raise ValueError("Raw corpus must be non-empty and contain 'doc_id', 'text', and 'split' columns. Re-run acquisition.")
    if set(frame["split"].dropna().unique()) != {"train", "test"}:
        raise ValueError("Raw corpus must contain both 'train' and 'test' split rows.")
    if frame["doc_id"].duplicated().any():
        raise ValueError("Raw corpus doc_id values must be unique.")
    if len(frame) < minimum_docs:
        raise ValueError(f"Raw corpus contains {len(frame):,} rows; at least {minimum_docs:,} are required for this run.")
    if exploratory:
        print("Using the professor-recommended 20 Newsgroups corpus.")
    print(f"Loaded {len(frame):,} documents for preprocessing...")
    cleaned = [clean_raw_string(value) for value in frame["text"]]
    tokenized = []
    for doc in tqdm(nlp.pipe(cleaned, batch_size=1000, n_process=args.n_process), total=len(cleaned), desc="Lemmatizing"):
        tokenized.append(_content_lemmas(doc))
    frame["tokens"] = tokenized
    frame["processed_text"] = [" ".join(tokens) for tokens in tokenized]
    before = len(frame)
    repeated_noise = frame["tokens"].map(
        lambda tokens: bool(tokens) and len(tokens) >= 30 and max(Counter(tokens).values()) / len(tokens) >= 0.30
    )
    dropped_repetitive = int(repeated_noise.sum())
    frame = frame.loc[~repeated_noise].copy()
    frame = frame.loc[frame["tokens"].map(len) >= 3].reset_index(drop=True)
    train_count = int((frame["split"] == "train").sum())
    test_count = int((frame["split"] == "test").sum())
    if train_count < minimum_docs or test_count == 0:
        raise ValueError(
            f"After preprocessing, train={train_count:,} and test={test_count:,}; this run requires "
            f"at least {minimum_docs:,} training documents and a non-empty test split."
        )
    frame[["doc_id", "split", "tokens", "processed_text"]].to_parquet(DATA_DIR / "processed_corpus.parquet", index=False)
    decisions = {
        "initial_documents": before,
        "retained_documents": len(frame),
        "dropped_short_documents": before - len(frame),
        "dropped_highly_repetitive_documents": dropped_repetitive,
        "minimum_required_documents": MIN_CORPUS_DOCS,
        "requirement_met": len(frame) >= MIN_CORPUS_DOCS,
        "run_type": "exploratory_under_minimum" if exploratory else "meets_assignment_minimum",
        "train_documents": train_count,
        "test_documents": test_count,
        "minimum_tokens_per_document": 3,
        "min_token_length": 3,
        "normalization": "HTML entity decoding and tag removal; lowercase; remove URLs, email addresses, standalone numbers, and non-ASCII letters; spaCy en_core_web_sm lemmatization; filter stopwords against surface forms and lemmas; remove documents with 30+ tokens where one token accounts for at least 30% of tokens",
        "custom_stopwords_added": sorted(CUSTOM_STOPWORDS),
        "stopwords_filtered": sorted(FILTERED_STOPWORDS),
    }
    (ARTIFACTS_DIR / "preprocessing_decisions.json").write_text(json.dumps(decisions, indent=2), encoding="utf-8")
    print(f"Preprocessing complete. Retained {len(frame):,} documents.")


sys.argv = ["step2_preprocess"]
main()


Using the professor-recommended 20 Newsgroups corpus.
Loaded 18,028 documents for preprocessing...


Lemmatizing: 100%|██████████| 18028/18028 [02:03<00:00, 145.72it/s]


Preprocessing complete. Retained 17,902 documents.


## Step 3 — Build text representations

Fit a unigram/bigram count vectorizer on training documents, then transform both splits to BoW and TF-IDF using the same training vocabulary. The held-out test split is only transformed, never used to fit the vocabulary or IDF weights.

In [4]:
# Step 3 implementation (step3_represent.py)
"""Build shared BoW and TF-IDF feature matrices and vocabulary reports."""

import json
import joblib
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer


def main() -> None:
    processed_path = DATA_DIR / "processed_corpus.parquet"
    if not processed_path.exists():
        raise FileNotFoundError(f"Processed corpus not found: {processed_path}. Run src.step2_preprocess first.")
    df = pd.read_parquet(processed_path)
    metadata_path = DATA_DIR / "dataset_metadata.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {}
    exploratory = bool(metadata.get("exploratory_under_minimum", False))
    minimum_train_docs = MIN_EXPLORATORY_TRAIN_DOCS if exploratory else MIN_CORPUS_DOCS
    if not {"processed_text", "split"}.issubset(df.columns) or df.empty:
        raise ValueError("Processed corpus must contain rows plus 'processed_text' and 'split' columns. Re-run preprocessing.")
    train_df = df.loc[df["split"] == "train"]
    test_df = df.loc[df["split"] == "test"]
    if len(train_df) < minimum_train_docs or test_df.empty:
        raise ValueError(
            f"Need at least {minimum_train_docs:,} training documents and a non-empty test split; "
            f"got {len(train_df):,} train and {len(test_df):,} test."
        )
    if exploratory:
        print(f"Vectorizing the selected corpus: {len(train_df):,} training and {len(test_df):,} held-out documents.")
    train_texts = train_df["processed_text"].tolist()
    test_texts = test_df["processed_text"].tolist()

    print(f"Fitting CountVectorizer (min_df={MIN_DF}, max_df={MAX_DF}, max_features={MAX_FEATURES}, ngram_range={NGRAM_RANGE})...")
    bow_vectorizer = CountVectorizer(
        min_df=MIN_DF,
        max_df=MAX_DF,
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        stop_words=sorted(VECTORIZER_STOPWORDS),
        token_pattern=r"(?u)\b[a-z]{3,}\b",
    )
    bow_train = bow_vectorizer.fit_transform(train_texts)
    bow_test = bow_vectorizer.transform(test_texts)

    print("Transforming BoW counts into TF-IDF matrix over the identical vocabulary...")
    tfidf_transformer = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
    tfidf_train = tfidf_transformer.fit_transform(bow_train)
    tfidf_test = tfidf_transformer.transform(bow_test)

    vocab = bow_vectorizer.get_feature_names_out().tolist()
    vocab_set = set(vocab)
    document_count = len(train_texts)
    # CountVectorizer exposes only retained terms in vocabulary_. Count document
    # frequency for every normalized token so the filtering deliverable is complete.
    from collections import Counter

    analyzer = bow_vectorizer.build_analyzer()
    dfs: Counter[str] = Counter()
    raw_unigram_dfs: Counter[str] = Counter()
    for text in train_texts:
        raw_terms = set(text.split())
        raw_unigram_dfs.update(raw_terms)
        dfs.update(set(analyzer(text)))
    max_df_count = MAX_DF * document_count if isinstance(MAX_DF, float) else MAX_DF
    filtered_rows = []
    for term in sorted(set(dfs) | VECTORIZER_STOPWORDS):
        df_count = raw_unigram_dfs.get(term, 0) if term in VECTORIZER_STOPWORDS else dfs[term]
        reasons = []
        if term in VECTORIZER_STOPWORDS:
            reasons.append("vectorizer_stopword")
        else:
            if df_count < MIN_DF:
                reasons.append("below_min_df")
            if df_count > max_df_count:
                reasons.append("above_max_df")
            if not reasons and term not in vocab_set:
                reasons.append("outside_max_features")
        if reasons:
            filtered_rows.append({"term": term, "document_frequency": df_count, "reason": ";".join(reasons)})

    # Save matrices and vectorizers
    sparse.save_npz(ARTIFACTS_DIR / "bow_train_matrix.npz", bow_train)
    sparse.save_npz(ARTIFACTS_DIR / "bow_test_matrix.npz", bow_test)
    sparse.save_npz(ARTIFACTS_DIR / "tfidf_train_matrix.npz", tfidf_train)
    sparse.save_npz(ARTIFACTS_DIR / "tfidf_test_matrix.npz", tfidf_test)
    joblib.dump(bow_vectorizer, ARTIFACTS_DIR / "bow_vectorizer.joblib")
    joblib.dump(tfidf_transformer, ARTIFACTS_DIR / "tfidf_transformer.joblib")

    vocab_report = {
        "document_count": document_count,
        "heldout_document_count": len(test_texts),
        "requirement_met": len(train_texts) + len(test_texts) >= MIN_CORPUS_DOCS,
        "retained_vocabulary_size": len(vocab),
        "total_filtered_words": len(filtered_rows),
        "filtered_words_by_reason": {
            "below_min_df": sum("below_min_df" in row["reason"] for row in filtered_rows),
            "above_max_df": sum("above_max_df" in row["reason"] for row in filtered_rows),
            "outside_max_features": sum("outside_max_features" in row["reason"] for row in filtered_rows),
            "vectorizer_stopword": sum("vectorizer_stopword" in row["reason"] for row in filtered_rows),
        },
        "min_df": MIN_DF,
        "max_df": MAX_DF,
        "max_features": MAX_FEATURES,
        "ngram_range": list(NGRAM_RANGE),
        "vectorizer_stopwords": sorted(VECTORIZER_STOPWORDS),
        "matrix_shape": list(bow_train.shape),
        "heldout_matrix_shape": list(bow_test.shape),
        "retained_vocabulary_sample": vocab[:200],
        "filtered_words_sample": [row["term"] for row in filtered_rows[:500]],
    }
    with open(ARTIFACTS_DIR / "filtered_vocabulary.json", "w", encoding="utf-8") as f:
        json.dump(vocab_report, f, indent=2)

    decisions_path = ARTIFACTS_DIR / "preprocessing_decisions.json"
    decisions = json.loads(decisions_path.read_text(encoding="utf-8")) if decisions_path.exists() else {}
    decisions["vectorizer_stopwords"] = sorted(VECTORIZER_STOPWORDS)
    decisions["vectorizer_filtering"] = {
        "min_df": MIN_DF,
        "max_df": MAX_DF,
        "max_features": MAX_FEATURES,
        "ngram_range": list(NGRAM_RANGE),
    }
    decisions_path.write_text(json.dumps(decisions, indent=2), encoding="utf-8")

    pd.Series(vocab, name="term").to_csv(ARTIFACTS_DIR / "vocabulary_terms.csv", index=False)
    pd.DataFrame(filtered_rows, columns=["term", "document_frequency", "reason"]).to_csv(
        ARTIFACTS_DIR / "filtered_words.csv", index=False
    )

    print(f"Saved training matrices {bow_train.shape} and held-out matrices {bow_test.shape}.")
    print(f"Retained vocabulary: {len(vocab):,} words | Filtered out: {len(filtered_rows):,} words.")


sys.argv = ["step3_represent"]
main()


Vectorizing the selected corpus: 10,756 training and 7,146 held-out documents.
Fitting CountVectorizer (min_df=20, max_df=0.35, max_features=15000, ngram_range=(1, 2))...
Transforming BoW counts into TF-IDF matrix over the identical vocabulary...
Saved training matrices (10756, 4560) and held-out matrices (7146, 4560).
Retained vocabulary: 4,560 words | Filtered out: 641,793 words.


## Step 4 — Train topic models

Fit LDA on BoW, NMF on BoW, and NMF on TF-IDF. Topic count, iteration limits, and random seed are set in the setup cell. Training can take several minutes, depending on the machine.

In [5]:
# Step 4 implementation (step4_train.py)
"""Fit reproducible LDA and NMF model configurations and record timings."""

import json
import platform
import time
import warnings
import joblib
import sklearn
import scipy
from scipy import sparse
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.exceptions import ConvergenceWarning


def main() -> None:
    metadata_path = DATA_DIR / "dataset_metadata.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {}
    exploratory = bool(metadata.get("exploratory_under_minimum", False))
    minimum_train_docs = MIN_EXPLORATORY_TRAIN_DOCS if exploratory else MIN_CORPUS_DOCS
    bow_path = ARTIFACTS_DIR / "bow_train_matrix.npz"
    tfidf_path = ARTIFACTS_DIR / "tfidf_train_matrix.npz"
    bow_test_path = ARTIFACTS_DIR / "bow_test_matrix.npz"
    tfidf_test_path = ARTIFACTS_DIR / "tfidf_test_matrix.npz"
    missing = [path.name for path in (bow_path, tfidf_path, bow_test_path, tfidf_test_path) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing feature matrices: {', '.join(missing)}. Run src.step3_represent first.")
    bow = sparse.load_npz(bow_path)
    tfidf = sparse.load_npz(tfidf_path)
    bow_test = sparse.load_npz(bow_test_path)
    tfidf_test = sparse.load_npz(tfidf_test_path)
    if bow.shape != tfidf.shape or bow.shape[0] == 0 or bow.shape[1] == 0:
        raise ValueError(f"BoW and TF-IDF matrices must have the same non-empty shape; got {bow.shape} and {tfidf.shape}.")
    if bow.shape[0] < minimum_train_docs:
        raise ValueError(f"Feature matrices contain {bow.shape[0]:,} documents; at least {minimum_train_docs:,} are required for this run.")
    if bow_test.shape != tfidf_test.shape or bow_test.shape[1] != bow.shape[1] or bow_test.shape[0] == 0:
        raise ValueError(f"Held-out matrices do not align with training features: train={bow.shape}, test={bow_test.shape}/{tfidf_test.shape}.")
    timings = {}
    training = {}
    models = {
        "lda_bow": (LatentDirichletAllocation(n_components=N_TOPICS, max_iter=MAX_ITER_LDA, learning_method="online", batch_size=2048, random_state=RANDOM_SEED, n_jobs=-1), bow),
        "nmf_bow": (NMF(n_components=N_TOPICS, init="nndsvda", solver="mu", beta_loss="kullback-leibler", max_iter=MAX_ITER_NMF, random_state=RANDOM_SEED), bow),
        "nmf_tfidf": (NMF(n_components=N_TOPICS, init="nndsvda", solver="mu", beta_loss="kullback-leibler", max_iter=MAX_ITER_NMF, random_state=RANDOM_SEED), tfidf),
    }
    for name, (model, matrix) in models.items():
        print(f"Training {name} on {matrix.shape} matrix...")
        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(matrix)
        timings[f"{name}_seconds"] = round(time.perf_counter() - start, 2)
        training[name] = {
            "n_iter": int(model.n_iter_),
            "max_iter": int(model.max_iter),
            "reached_iteration_limit": int(model.n_iter_) >= int(model.max_iter),
            "warnings": [str(item.message) for item in caught],
        }
        joblib.dump(model, ARTIFACTS_DIR / f"{name}_model.joblib")
        print(f"  completed in {timings[f'{name}_seconds']} seconds")
    report = {
        "training_timings_seconds": timings,
        "training_diagnostics": training,
        "experiment": {
            "topic_count": N_TOPICS,
            "ngram_range": list(NGRAM_RANGE),
            "vectorizer_stopwords": sorted(VECTORIZER_STOPWORDS),
            "random_seed": RANDOM_SEED,
            "lda_max_iter": MAX_ITER_LDA,
            "nmf_max_iter": MAX_ITER_NMF,
            "nmf_solver": "mu",
            "nmf_beta_loss": "kullback-leibler",
            "document_count": int(bow.shape[0]),
            "heldout_document_count": int(bow_test.shape[0]),
            "requirement_met": int(bow.shape[0] + bow_test.shape[0]) >= MIN_CORPUS_DOCS,
            "run_type": "exploratory_under_minimum" if exploratory else "meets_assignment_minimum",
            "vocabulary_size": int(bow.shape[1]),
            "python_version": platform.python_version(),
            "scikit_learn_version": sklearn.__version__,
            "scipy_version": scipy.__version__,
        },
    }
    (ARTIFACTS_DIR / "training_timings.json").write_text(json.dumps(report, indent=2), encoding="utf-8")


sys.argv = ["step4_train"]
main()


Training lda_bow on (10756, 4560) matrix...
  completed in 17.11 seconds
Training nmf_bow on (10756, 4560) matrix...
  completed in 18.77 seconds
Training nmf_tfidf on (10756, 4560) matrix...
  completed in 13.73 seconds


## Step 5 — Evaluate and export results

Evaluate topic coherence and diversity on held-out text, compare topics, and use held-out labels only for external alignment diagnostics. Review generated CSV and JSON results in `artifacts/`; category alignment does not replace human review of topic meaning.

In [6]:
# Step 5 implementation (step5_evaluate.py)
"""Evaluate topic word lists and export metrics without inventing results."""

import json
import joblib
import numpy as np
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse


def extract_top_words(model, vocabulary: list[str]) -> list[list[str]]:
    return [[vocabulary[i] for i in row.argsort()[-TOP_N_WORDS:][::-1]] for row in model.components_]


def topic_diversity(topics: list[list[str]]) -> float:
    words = [word for topic in topics for word in topic]
    return len(set(words)) / len(words) if words else 0.0


def topic_overlap_metrics(topics: list[list[str]], components: np.ndarray) -> dict[str, float]:
    """Quantify model-topic redundancy with top-word Jaccard and component cosine."""
    if len(topics) < 2:
        return {"mean_top_word_jaccard_similarity": 0.0, "mean_component_cosine_similarity": 0.0, "max_component_cosine_similarity": 0.0}
    jaccard = []
    for left in range(len(topics)):
        for right in range(left + 1, len(topics)):
            left_words, right_words = set(topics[left]), set(topics[right])
            union = left_words | right_words
            jaccard.append(len(left_words & right_words) / len(union) if union else 0.0)
    normalized = components / np.maximum(np.linalg.norm(components, axis=1, keepdims=True), 1e-12)
    similarities = cosine_similarity(normalized)
    upper = similarities[np.triu_indices_from(similarities, k=1)]
    return {
        "mean_top_word_jaccard_similarity": round(float(np.mean(jaccard)), 4),
        "mean_component_cosine_similarity": round(float(np.mean(upper)), 4),
        "max_component_cosine_similarity": round(float(np.max(upper)), 4),
    }


def main() -> None:
    processed_path = DATA_DIR / "processed_corpus.parquet"
    if not processed_path.exists():
        raise FileNotFoundError(f"Processed corpus not found: {processed_path}. Run src.step2_preprocess first.")
    frame = pd.read_parquet(processed_path)
    metadata_path = DATA_DIR / "dataset_metadata.json"
    dataset_metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {}
    exploratory = bool(dataset_metadata.get("exploratory_under_minimum", False))
    minimum_train_docs = MIN_EXPLORATORY_TRAIN_DOCS if exploratory else MIN_CORPUS_DOCS
    if frame.empty or not {"doc_id", "split", "tokens", "processed_text"}.issubset(frame.columns):
        raise ValueError("Processed corpus must contain doc_id, split, tokens, and processed_text columns.")
    train_frame = frame.loc[frame["split"] == "train"]
    test_frame = frame.loc[frame["split"] == "test"].reset_index(drop=True)
    if len(train_frame) < minimum_train_docs or test_frame.empty:
        raise ValueError(f"Need at least {minimum_train_docs:,} training documents and a non-empty held-out test split.")
    # Match CountVectorizer's unigram + bigram vocabulary during coherence
    # evaluation so phrase topics (e.g. "space_shuttle") are evaluated as phrases.
    texts = []
    for tokens in test_frame["tokens"]:
        tokens = [token for token in tokens if token not in VECTORIZER_STOPWORDS]
        phrase_aware = []
        for index, token in enumerate(tokens):
            phrase_aware.append(token)
            if index + 1 < len(tokens):
                phrase_aware.append(f"{token} {tokens[index + 1]}")
        texts.append(phrase_aware)
    vectorizer = joblib.load(ARTIFACTS_DIR / "bow_vectorizer.joblib")
    vocabulary = vectorizer.get_feature_names_out().tolist()
    vocabulary_set = set(vocabulary)
    dictionary = Dictionary([[term for term in text if term in vocabulary_set] for text in texts])
    corpus = [dictionary.doc2bow(doc) for doc in texts]
    model_files = {
        "LDA_BoW (Controlled & Practical)": ("lda_bow_model.joblib", "bow_test_matrix.npz"),
        "NMF_BoW (Controlled)": ("nmf_bow_model.joblib", "bow_test_matrix.npz"),
        "NMF_TFIDF (Practical)": ("nmf_tfidf_model.joblib", "tfidf_test_matrix.npz"),
    }
    missing_models = [filename for pair in model_files.values() for filename in pair if not (ARTIFACTS_DIR / filename).exists()]
    timings_path = ARTIFACTS_DIR / "training_timings.json"
    if missing_models or not timings_path.exists():
        missing = missing_models + ([] if timings_path.exists() else [timings_path.name])
        raise FileNotFoundError(f"Missing training artifacts: {', '.join(missing)}. Run src.step4_train first.")
    training_report = json.loads(timings_path.read_text(encoding="utf-8"))
    interpretations_path = BASE_DIR / "topic_interpretations.json"
    interpretations_path.write_text(json.dumps(INTERPRETATIONS, indent=2, ensure_ascii=False), encoding="utf-8")
    interpretations = INTERPRETATIONS
    label_path = DATA_DIR / "isolated_labels_for_evaluation.parquet"
    if not label_path.exists():
        label_path = DATA_DIR / "isolated_labels_do_not_use.parquet"
    heldout_labels = None
    if dataset_metadata.get("label_names") and label_path.exists():
        label_frame = pd.read_parquet(label_path)
        heldout_labels = label_frame.set_index("doc_id")["label"].reindex(test_frame["doc_id"]).to_numpy()
        if len(heldout_labels) != len(test_frame) or pd.isna(heldout_labels).any():
            raise ValueError("Could not align held-out labels with test document IDs.")
    category_names = dataset_metadata.get("label_names", [])
    summary, rows = {}, []
    labelled_topics = 0
    for label, (model_filename, matrix_filename) in model_files.items():
        model = joblib.load(ARTIFACTS_DIR / model_filename)
        test_matrix = sparse.load_npz(ARTIFACTS_DIR / matrix_filename)
        if test_matrix.shape != (len(test_frame), len(vocabulary)):
            raise ValueError(f"Held-out matrix for {label} does not align with test documents and fitted vocabulary.")
        topics = extract_top_words(model, vocabulary)
        cv_model = CoherenceModel(topics=topics, texts=texts, dictionary=dictionary, coherence="c_v", processes=1)
        cv = float(cv_model.get_coherence())
        per_topic = [float(score) for score in cv_model.get_coherence_per_topic()]
        umass = float(CoherenceModel(topics=topics, corpus=corpus, dictionary=dictionary, coherence="u_mass").get_coherence())
        diversity = topic_diversity(topics)
        overlap = topic_overlap_metrics(topics, np.asarray(model.components_, dtype=np.float64))
        scores = model.transform(test_matrix)
        signal = np.asarray(scores.sum(axis=1)).ravel() > 1e-12
        predicted_topics = np.argmax(scores[signal], axis=1)
        extrinsic = None
        if heldout_labels is not None and signal.any():
            valid_labels = np.asarray(heldout_labels)[signal]
            extrinsic = {
                "documents_scored": int(signal.sum()),
                "adjusted_rand_index": round(float(adjusted_rand_score(valid_labels, predicted_topics)), 4),
                "normalized_mutual_information": round(float(normalized_mutual_info_score(valid_labels, predicted_topics)), 4),
            }
            topic_summary = {}
            for topic_id in range(model.n_components):
                topic_mask = predicted_topics == topic_id
                if not topic_mask.any():
                    continue
                labels, counts = np.unique(valid_labels[topic_mask], return_counts=True)
                majority_index = int(np.argmax(counts))
                majority_label = int(labels[majority_index])
                majority_name = category_names[majority_label] if majority_label < len(category_names) else str(majority_label)
                topic_summary[f"Topic_{topic_id + 1}"] = {
                    "documents": int(topic_mask.sum()),
                    "dominant_category": majority_name,
                    "category_purity": round(float(counts[majority_index] / counts.sum()), 4),
                }
            extrinsic["majority_category_purity"] = round(
                sum(item["documents"] * item["category_purity"] for item in topic_summary.values()) / int(signal.sum()), 4
            )
            extrinsic["topic_category_summary"] = topic_summary
            category_topic_summary = {}
            for category_id, category_name in enumerate(category_names):
                category_mask = valid_labels == category_id
                if not category_mask.any():
                    continue
                topic_counts = np.bincount(predicted_topics[category_mask], minlength=model.n_components)
                top_topic = int(topic_counts.argmax())
                distribution = topic_counts / topic_counts.sum()
                positive = distribution[distribution > 0]
                entropy = float(-(positive * np.log(positive)).sum())
                category_topic_summary[category_name] = {
                    "documents": int(category_mask.sum()),
                    "dominant_topic": f"Topic_{top_topic + 1}",
                    "dominant_topic_share": round(float(distribution[top_topic]), 4),
                    "topic_entropy": round(entropy, 4),
                    "topic_distribution": {
                        f"Topic_{i + 1}": round(float(share), 4)
                        for i, share in enumerate(distribution) if share > 0
                    },
                }
            extrinsic["category_topic_summary"] = category_topic_summary
        summary[label] = {
            "coherence_c_v": round(cv, 4),
            "coherence_u_mass": round(umass, 4),
            "topic_diversity": round(diversity, 4),
            "redundancy_rate": round(1 - diversity, 4),
            "topic_overlap": overlap,
            "coherence_c_v_per_topic": [round(x, 4) for x in per_topic],
            "topics": {f"Topic_{i + 1}": words for i, words in enumerate(topics)},
            "external_alignment": extrinsic,
        }
        interpretation_records = {}
        for i, words in enumerate(topics):
            word_key = ", ".join(words)
            interpretation = interpretations.get(label, {}).get(word_key, {})
            manual_label = interpretation.get("label", "Pending human interpretation")
            rationale = interpretation.get("rationale", "")
            labelled_topics += manual_label != "Pending human interpretation"
            interpretation_records[f"Topic_{i + 1}"] = {"label": manual_label, "rationale": rationale}
            rows.append({
                "Model": label,
                "Topic_ID": f"Topic {i + 1}",
                "C_v_Coherence": round(per_topic[i], 4),
                "Top_10_Words": word_key,
                "Manual_Interpretation_Label": manual_label,
                "Interpretation_Rationale": rationale,
            })
        summary[label]["interpretations"] = interpretation_records
    summary["evaluation_metadata"] = {
        "training_documents": len(train_frame),
        "heldout_documents_evaluated": len(texts),
        "dataset_source": dataset_metadata.get("source"),
        "minimum_documents_required": MIN_CORPUS_DOCS,
        "requirement_met": len(frame) >= MIN_CORPUS_DOCS,
        "run_type": "exploratory_under_minimum" if exploratory else "meets_assignment_minimum",
        "vocabulary_size": len(vocabulary),
        "top_words_per_topic": TOP_N_WORDS,
        "coherence_corpus": "Held-out test documents; phrase-aware sliding windows; vectorizer and models were fit on training only.",
        "external_label_source": f"{dataset_metadata.get('source', 'Dataset')} labels from the held-out test split only; labels are never used for preprocessing or model fitting." if heldout_labels is not None else None,
        "category_alignment_caveat": "ARI/NMI measure alignment with dataset categories, not semantic topic quality; labels are evaluation-only and never used for fitting.",
        "model_selection_note": "Configurations were explored during this project. Treat these test-split results as exploratory comparisons, not an untouched final benchmark.",
        "nmf_scores_note": "NMF document weights are activations, not calibrated probabilities.",
        "manually_interpreted_topics": labelled_topics,
        "interpretations_source": interpretations_path.name if interpretations_path.exists() else None,
    }
    summary["training_timings_seconds"] = training_report.get("training_timings_seconds", training_report)
    if "training_diagnostics" in training_report:
        summary["training_diagnostics"] = training_report["training_diagnostics"]
    if "experiment" in training_report:
        summary["experiment"] = training_report["experiment"]
    (ARTIFACTS_DIR / "evaluation_results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    pd.DataFrame(rows).to_csv(ARTIFACTS_DIR / "topics_top10.csv", index=False)
    print("Evaluation results written to artifacts/evaluation_results.json and artifacts/topics_top10.csv")


sys.argv = ["step5_evaluate"]
main()


Evaluation results written to artifacts/evaluation_results.json and artifacts/topics_top10.csv


## Step 5b — Compare model performance visually

Load the saved evaluation scores, compare quality, external category alignment, diversity, and training time, and inspect per-topic coherence. Cᵥ and U-Mass use different scales, so they are shown separately. External alignment uses labels held out from model fitting.


In [ ]:
# Compare the saved held-out evaluation metrics and render simple SVG charts (no extra plotting package needed).
import json
import pandas as pd
from IPython.display import SVG, display

evaluation_path = ARTIFACTS_DIR / "evaluation_results.json"
evaluation = json.loads(evaluation_path.read_text(encoding="utf-8"))
model_names = [
    "LDA_BoW (Controlled & Practical)",
    "NMF_BoW (Controlled)",
    "NMF_TFIDF (Practical)",
]
labels = ["LDA · BoW", "NMF · BoW", "NMF · TF-IDF"]
rows = []
for model, label in zip(model_names, labels):
    result = evaluation[model]
    alignment = result.get("external_alignment") or {}
    rows.append({
        "Model": label,
        "C_v coherence": result["coherence_c_v"],
        "U-Mass coherence": result["coherence_u_mass"],
        "Topic diversity": result["topic_diversity"],
        "Redundancy": result["redundancy_rate"],
        "ARI": alignment.get("adjusted_rand_index"),
        "NMI": alignment.get("normalized_mutual_information"),
        "Category purity": alignment.get("majority_category_purity"),
        "Training time (s)": evaluation.get("training_timings_seconds", {}).get({
            model_names[0]: "lda_bow_seconds",
            model_names[1]: "nmf_bow_seconds",
            model_names[2]: "nmf_tfidf_seconds",
        }[model]),
    })
metrics_df = pd.DataFrame(rows).set_index("Model")
display(metrics_df.style.format({
    "C_v coherence": "{:.4f}", "U-Mass coherence": "{:.4f}",
    "Topic diversity": "{:.1%}", "Redundancy": "{:.1%}",
    "ARI": "{:.4f}", "NMI": "{:.4f}",
    "Category purity": "{:.1%}", "Training time (s)": "{:.2f}",
}))

def metric_svg(title, subtitle, metric, maximum, formatter):
    colors = ["#355C7D", "#2A7F75", "#D17A32"]
    chunks = [
        '<svg xmlns="http://www.w3.org/2000/svg" width="900" height="300" viewBox="0 0 900 300">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<style>text{font-family:Arial,sans-serif;fill:#19232d}</style>',
        f'<text x="30" y="38" font-size="23" font-weight="bold">{title}</text>',
        f'<text x="30" y="64" font-size="14" fill="#66727d">{subtitle}</text>',
    ]
    for i, (label, value) in enumerate(zip(metrics_df.index, metrics_df[metric])):
        y = 100 + i * 58
        width = 570 * max(0, min(1, float(value) / maximum))
        chunks.extend([
            f'<text x="30" y="{y+21}" font-size="16">{label}</text>',
            f'<rect x="220" y="{y}" width="570" height="30" rx="3" fill="#eef1f3"/>',
            f'<rect x="220" y="{y}" width="{width:.1f}" height="30" rx="3" fill="{colors[i]}"/>',
            f'<text x="{230+width:.1f}" y="{y+21}" font-size="15" font-weight="bold">{formatter(float(value))}</text>',
        ])
    chunks.append('</svg>')
    return SVG("\n".join(chunks))

display(metric_svg("Topic coherence (Cᵥ)", "Higher is better", "C_v coherence", 0.75, lambda x: f"{x:.4f}"))
display(metric_svg("Category alignment (ARI)", "External diagnostic; labels were not used for fitting", "ARI", 0.35, lambda x: f"{x:.4f}"))
display(metric_svg("Topic diversity", "Fraction of unique words across top-word lists", "Topic diversity", 1.0, lambda x: f"{x:.1%}"))
display(metric_svg("Training time", "This run; hardware dependent, lower is faster", "Training time (s)", 50, lambda x: f"{x:.2f} s"))

# C_v distribution across each model's topics reveals weak or mixed topics hidden by the mean.
per_topic = pd.DataFrame({
    label: evaluation[model]["coherence_c_v_per_topic"]
    for model, label in zip(model_names, labels)
}, index=[f"Topic {i}" for i in range(1, 21)])
display(per_topic.describe().T[["mean", "min", "max"]].style.format("{:.4f}"))


## Output files

- `data/raw_corpus.parquet` and `data/processed_corpus.parquet` hold the unlabeled corpus at two processing stages.
- `data/dataset_metadata.json` records the source and split sizes.
- `data/isolated_labels_for_evaluation.parquet` contains labels, which fitting stages do not read.
- `artifacts/` contains sparse matrices, fitted vectorizers and models, diagnostics, topic lists, and evaluation metrics.

If you change `DATA_SOURCE`, rerun all six stage cells in order so outputs from different datasets are not mixed. The original command-line scripts remain available under `src/` and produce the same outputs.

## Example documents for inference

These three short files are qualitative smoke tests, each written to emphasize one distinct 20 Newsgroups subject. Run the cell to score every document with all three trained models. A top assignment shows that the pipeline can recognize a clear example; these hand-written examples do not replace held-out evaluation or establish model accuracy. NMF values are normalized activations, not calibrated probabilities.

In [7]:
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

# Load the fitted artifacts produced by Steps 3–5.
bow_vectorizer = joblib.load(ARTIFACTS_DIR / "bow_vectorizer.joblib")
tfidf_transformer = joblib.load(ARTIFACTS_DIR / "tfidf_transformer.joblib")
evaluation = json.loads((ARTIFACTS_DIR / "evaluation_results.json").read_text(encoding="utf-8"))
model_specs = [
    ("LDA_BoW (Controlled & Practical)", "lda_bow_model.joblib", "bow"),
    ("NMF_BoW (Controlled)", "nmf_bow_model.joblib", "bow"),
    ("NMF_TFIDF (Practical)", "nmf_tfidf_model.joblib", "tfidf"),
]
test_dir = BASE_DIR / "test_documents"
rows = []

for text_path in sorted(test_dir.glob("*.txt")):
    tokens = preprocess_single_document(text_path.read_text(encoding="utf-8"))
    bow = bow_vectorizer.transform([" ".join(tokens)])
    for model_name, model_file, representation in model_specs:
        model = joblib.load(ARTIFACTS_DIR / model_file)
        matrix = tfidf_transformer.transform(bow) if representation == "tfidf" else bow
        scores = np.asarray(model.transform(matrix))[0]
        total = float(scores.sum())
        shares = scores / total if total else scores
        topic_index = int(np.argmax(shares))
        topic_id = f"Topic_{topic_index + 1}"
        topic = evaluation[model_name]["topics"][topic_id]
        interpretation = evaluation[model_name]["interpretations"][topic_id]
        rows.append({
            "File": text_path.name,
            "Model": model_name,
            "Top topic": f"Topic {topic_index + 1}",
            "Interpretation": interpretation["label"],
            "Normalized score": round(float(shares[topic_index]), 3),
            "Top words": ", ".join(topic),
        })

display(pd.DataFrame(rows))


,File,Model,Top topic,Interpretation,Normalized score,Top words
0,01_cryptography_encryption.txt,LDA_BoW (Controlled & Practical),Topic 15,Encryption policy and firearms law (mixed),0.807,"key, gun, law, chip, encryption, right, public..."
1,01_cryptography_encryption.txt,NMF_BoW (Controlled),Topic 19,"Cryptography, computer security, and privacy p...",0.880,"key, chip, encryption, clipper, government, sy..."
2,01_cryptography_encryption.txt,NMF_TFIDF (Practical),Topic 5,"Cryptography, encryption, and privacy policy",0.822,"key, chip, encryption, clipper, government, al..."
3,02_space_exploration.txt,LDA_BoW (Controlled & Practical),Topic 8,Space exploration and NASA research,0.932,"space, launch, system, datum, nasa, informatio..."
4,02_space_exploration.txt,NMF_BoW (Controlled),Topic 10,"Spaceflight, satellites, and NASA missions",0.819,"space, launch, nasa, satellite, earth, orbit, ..."
5,02_space_exploration.txt,NMF_TFIDF (Practical),Topic 17,"Space missions, NASA, and project costs",0.740,"space, money, cost, air, launch, spend, nasa, ..."
6,03_hockey_game.txt,LDA_BoW (Controlled & Practical),Topic 19,"Team sports, especially hockey",0.922,"game, team, play, win, player, season, hockey,..."
7,03_hockey_game.txt,NMF_BoW (Controlled),Topic 7,"Team sports, especially hockey (baseball overlap)",0.783,"game, team, play, win, player, season, league,..."
8,03_hockey_game.txt,NMF_TFIDF (Practical),Topic 3,Team sports (baseball and hockey grouped toget...,0.836,"game, team, play, player, win, season, fan, ho..."
